# Kaggle Regression Competition — Predict `BeatsPerMinute`

**Evaluation Metric:** Root Mean Squared Error (RMSE)

## Pipeline Overview
1. Data Loading & Initial Inspection
2. Train-Validation Split (before any transformation)
3. Missing Value Handling & Preprocessing (anti-leakage)
4. Automated Feature Selection
5. Model Training & Evaluation (RF, XGBoost, LightGBM)
6. Best Model Selection & Retrain on Full Data
7. Submission Generation

---
## 0. Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel, mutual_info_regression
from sklearn.metrics import mean_squared_error

# Models
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports successful.')

All imports successful.


---
## 1. Data Loading & Initial Inspection

In [2]:
# Load datasets
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test  shape: {test_df.shape}')
print(f'Sample submission shape: {sample_sub.shape}')
print()
train_df.head()

Train shape: (524164, 11)
Test  shape: (174722, 10)
Sample submission shape: (174722, 2)



,id,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
0,0,0.603610,-7.636942,0.023500,0.000005,0.000001,0.051385,0.409866,290715.6450,0.826267,147.53020
1,1,0.639451,-16.267598,0.071520,0.444929,0.349414,0.170522,0.651010,164519.5174,0.145400,136.15963
2,2,0.514538,-15.953575,0.110715,0.173699,0.453814,0.029576,0.423865,174495.5667,0.624667,55.31989
3,3,0.734463,-1.357000,0.052965,0.001651,0.159717,0.086366,0.278745,225567.4651,0.487467,147.91212
4,4,0.532968,-13.056437,0.023500,0.068687,0.000001,0.331345,0.477769,213960.6789,0.947333,89.58511


In [3]:
# Quick data overview
print('--- Data types ---')
print(train_df.dtypes)
print()
print('--- Descriptive statistics ---')
train_df.describe()

--- Data types ---
id                             int64
RhythmScore                  float64
AudioLoudness                float64
VocalContent                 float64
AcousticQuality              float64
InstrumentalScore            float64
LivePerformanceLikelihood    float64
MoodScore                    float64
TrackDurationMs              float64
Energy                       float64
BeatsPerMinute               float64
dtype: object

--- Descriptive statistics ---


,id,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
count,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000
mean,262081.500000,0.632843,-8.379014,0.074443,0.262913,0.117690,0.178398,0.555843,241903.692949,0.500923,119.034899
std,151313.257587,0.156899,4.616221,0.049939,0.223120,0.131845,0.118186,0.225480,59326.601501,0.289952,26.468077
min,0.000000,0.076900,-27.509725,0.023500,0.000005,0.000001,0.024300,0.025600,63973.000000,0.000067,46.718000
25%,131040.750000,0.515850,-11.551933,0.023500,0.069413,0.000001,0.077637,0.403921,207099.876625,0.254933,101.070410
50%,262081.500000,0.634686,-8.252499,0.066425,0.242502,0.074247,0.166327,0.564817,243684.058150,0.511800,118.747660
75%,393122.250000,0.739179,-4.912298,0.107343,0.396957,0.204065,0.268946,0.716633,281851.658500,0.746000,136.686590
max,524163.000000,0.975000,-1.357000,0.256401,0.995000,0.869258,0.599924,0.978000,464723.228100,1.000000,206.037000


In [4]:
# Separate target and IDs
TARGET = 'BeatsPerMinute'
ID_COL = 'id'

# Extract IDs
train_ids = train_df[ID_COL].copy()
test_ids  = test_df[ID_COL].copy()

# Target
y_full = train_df[TARGET].copy()

# Features (drop ID and target from train, drop ID from test)
X_full = train_df.drop(columns=[ID_COL, TARGET])
X_test = test_df.drop(columns=[ID_COL])

print(f'Features shape (train): {X_full.shape}')
print(f'Features shape (test) : {X_test.shape}')
print(f'Target shape          : {y_full.shape}')

Features shape (train): (524164, 9)
Features shape (test) : (174722, 9)
Target shape          : (524164,)


---
## 2. Train-Validation Split (Before Any Transformation)

This is done **before** any preprocessing to prevent data leakage.

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f'Training   set: {X_train.shape}')
print(f'Validation set: {X_val.shape}')

Training   set: (419331, 9)
Validation set: (104833, 9)


---
## 3. Missing Value Handling & Preprocessing

We build an `sklearn` `ColumnTransformer` that:
- **Numerical features:** median imputation → StandardScaler
- **Categorical features:** most-frequent imputation → OneHotEncoder (low cardinality) or capped OneHotEncoder (high cardinality)

Everything is fit **only** on `X_train`.

In [6]:
# Identify column types
numerical_cols   = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numerical features   ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

# Check missing values
print('\n--- Missing values in training split ---')
missing_train = X_train.isnull().sum()
print(missing_train[missing_train > 0] if missing_train.sum() > 0 else 'No missing values.')

Numerical features   (9): ['RhythmScore', 'AudioLoudness', 'VocalContent', 'AcousticQuality', 'InstrumentalScore', 'LivePerformanceLikelihood', 'MoodScore', 'TrackDurationMs', 'Energy']
Categorical features (0): []

--- Missing values in training split ---
No missing values.


In [7]:
# ---- Build preprocessing pipelines ----

# Numerical pipeline: median imputation + standard scaling
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical pipeline (if any categorical columns exist)
# Separate low-cardinality (<= 15 unique) and high-cardinality (> 15 unique)
if categorical_cols:
    low_card_cols  = [c for c in categorical_cols if X_train[c].nunique() <= 15]
    high_card_cols = [c for c in categorical_cols if X_train[c].nunique() > 15]
    print(f'Low-cardinality  categorical: {low_card_cols}')
    print(f'High-cardinality categorical: {high_card_cols}')
else:
    low_card_cols  = []
    high_card_cols = []
    print('No categorical columns detected.')

cat_low_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_high_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Capped OneHotEncoder for high-cardinality (top 20 categories)
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=20))
])

# Assemble ColumnTransformer
transformers = [('num', num_pipeline, numerical_cols)]
if low_card_cols:
    transformers.append(('cat_low',  cat_low_pipeline,  low_card_cols))
if high_card_cols:
    transformers.append(('cat_high', cat_high_pipeline, high_card_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

print('\nPreprocessor built successfully.')

No categorical columns detected.

Preprocessor built successfully.


In [8]:
# Fit ONLY on training split, then transform train / val / test
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

print(f'Processed training   shape: {X_train_proc.shape}')
print(f'Processed validation shape: {X_val_proc.shape}')
print(f'Processed test       shape: {X_test_proc.shape}')

Processed training   shape: (419331, 9)
Processed validation shape: (104833, 9)
Processed test       shape: (174722, 9)


---
## 4. Automated Feature Selection

We use `SelectFromModel` with a lightweight `RandomForestRegressor` as the
meta-estimator to keep only the most predictive features based on
feature importances.

In [9]:
# Fit a small RF to derive feature importances
selector_estimator = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

selector = SelectFromModel(
    estimator=selector_estimator,
    threshold='median'  # keep features with importance >= median
)

# Fit on training data ONLY
selector.fit(X_train_proc, y_train)

# Transform all sets
X_train_sel = selector.transform(X_train_proc)
X_val_sel   = selector.transform(X_val_proc)
X_test_sel  = selector.transform(X_test_proc)

n_selected = X_train_sel.shape[1]
n_original = X_train_proc.shape[1]

print(f'Features selected: {n_selected} / {n_original}')
print(f'Training   shape after selection: {X_train_sel.shape}')
print(f'Validation shape after selection: {X_val_sel.shape}')
print(f'Test       shape after selection: {X_test_sel.shape}')

Features selected: 5 / 9
Training   shape after selection: (419331, 5)
Validation shape after selection: (104833, 5)
Test       shape after selection: (174722, 5)


In [10]:
# Show feature importances from the selector
importances = selector.estimator_.feature_importances_
mask = selector.get_support()

print('Feature importances (selected features marked with [x]):')
for i, (imp, selected) in enumerate(zip(importances, mask)):
    mark = 'x' if selected else ' '
    print(f'  [{mark}] Feature {i:3d}: {imp:.6f}')

Feature importances (selected features marked with [x]):
  [x] Feature   0: 0.131356
  [ ] Feature   1: 0.104050
  [x] Feature   2: 0.115197
  [ ] Feature   3: 0.088456
  [ ] Feature   4: 0.086210
  [x] Feature   5: 0.105008
  [ ] Feature   6: 0.097109
  [x] Feature   7: 0.145608
  [x] Feature   8: 0.127008


---
## 5. Model Training & Evaluation

We train three tree-based regressors and evaluate each on the held-out validation set.

In [11]:
def rmse(y_true, y_pred):
    """Compute Root Mean Squared Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [12]:
# ---- Model 1: Random Forest Regressor ----
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train_sel, y_train)
rf_val_pred = rf_model.predict(X_val_sel)
rf_rmse = rmse(y_val, rf_val_pred)
print(f'Random Forest  -- Validation RMSE: {rf_rmse:.5f}')

Random Forest  -- Validation RMSE: 26.76686


In [13]:
# ---- Model 2: XGBoost Regressor ----
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=50
)
xgb_model.fit(
    X_train_sel, y_train,
    eval_set=[(X_val_sel, y_val)],
    verbose=False
)
xgb_val_pred = xgb_model.predict(X_val_sel)
xgb_rmse = rmse(y_val, xgb_val_pred)
print(f'XGBoost        -- Validation RMSE: {xgb_rmse:.5f}')

XGBoost        -- Validation RMSE: 26.44413


In [14]:
# ---- Model 3: LightGBM Regressor ----
lgbm_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)
lgbm_model.fit(
    X_train_sel, y_train,
    eval_set=[(X_val_sel, y_val)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(period=0)
    ]
)
lgbm_val_pred = lgbm_model.predict(X_val_sel)
lgbm_rmse = rmse(y_val, lgbm_val_pred)
print(f'LightGBM       -- Validation RMSE: {lgbm_rmse:.5f}')

LightGBM       -- Validation RMSE: 26.44487


In [15]:
# ---- Summary of all models ----
print('=' * 50)
print('  MODEL COMPARISON -- Validation RMSE')
print('=' * 50)
results = {
    'Random Forest': rf_rmse,
    'XGBoost':       xgb_rmse,
    'LightGBM':      lgbm_rmse
}
for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f'  {name:<15s}  RMSE = {score:.5f}')
print('=' * 50)

  MODEL COMPARISON -- Validation RMSE
  XGBoost          RMSE = 26.44413
  LightGBM         RMSE = 26.44487
  Random Forest    RMSE = 26.76686


---
## 6. Best Model Selection & Retrain on Full Data

We pick the model with the **lowest validation RMSE**, refit the
preprocessor + selector on the **entire** `train.csv` (train + val),
retrain the best model, and generate test predictions.

In [16]:
# Identify the best model
best_name = min(results, key=results.get)
best_rmse = results[best_name]
print(f'\n>>> Best model: {best_name} (Val RMSE = {best_rmse:.5f})')

# Map name -> model class + hyperparams for fresh retrain
model_registry = {
    'Random Forest': RandomForestRegressor(
        n_estimators=500, max_depth=None, min_samples_split=5,
        min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': XGBRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=-1,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=-1
    )
}

final_model = model_registry[best_name]


>>> Best model: XGBoost (Val RMSE = 26.44413)


In [17]:
# Refit preprocessor on the FULL training data (train + val)
X_full_proc = preprocessor.fit_transform(X_full)
X_test_proc_final = preprocessor.transform(X_test)

# Refit selector on the full training data
selector.fit(X_full_proc, y_full)
X_full_sel = selector.transform(X_full_proc)
X_test_sel_final = selector.transform(X_test_proc_final)

print(f'Full training set (selected features): {X_full_sel.shape}')
print(f'Test set          (selected features): {X_test_sel_final.shape}')

Full training set (selected features): (524164, 5)
Test set          (selected features): (174722, 5)


In [18]:
# Retrain the best model on the entire training data
final_model.fit(X_full_sel, y_full)
print(f'{best_name} retrained on full training data ({X_full_sel.shape[0]} samples).')

# Generate predictions on the test set
test_predictions = final_model.predict(X_test_sel_final)
print(f'Test predictions generated: {test_predictions.shape}')
print(f'Prediction range: [{test_predictions.min():.2f}, {test_predictions.max():.2f}]')

XGBoost retrained on full training data (524164 samples).
Test predictions generated: (174722,)
Prediction range: [95.98, 143.82]


---
## 7. Submission Generation

In [19]:
# Create submission dataframe matching sample_submission.csv format
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: test_predictions
})

# Sanity checks
assert submission.shape[0] == sample_sub.shape[0], \
    f'Row count mismatch: {submission.shape[0]} vs {sample_sub.shape[0]}'
assert list(submission.columns) == list(sample_sub.columns), \
    f'Column mismatch: {list(submission.columns)} vs {list(sample_sub.columns)}'

# Export
submission.to_csv('submission.csv', index=False)
print('submission.csv saved successfully!')
print(f'Shape: {submission.shape}')
submission.head(10)

submission.csv saved successfully!
Shape: (174722, 2)


,id,BeatsPerMinute
0,524164,117.991371
1,524165,118.638557
2,524166,117.427834
3,524167,119.156540
4,524168,119.068413
5,524169,119.021477
6,524170,114.880707
7,524171,118.133667
8,524172,119.551453
9,524173,117.229118


In [20]:
print('\n' + '=' * 50)
print('  PIPELINE COMPLETE')
print('=' * 50)
print(f'  Best model      : {best_name}')
print(f'  Validation RMSE : {best_rmse:.5f}')
print(f'  Submission rows : {submission.shape[0]}')
print(f'  Output file     : submission.csv')
print('=' * 50)


  PIPELINE COMPLETE
  Best model      : XGBoost
  Validation RMSE : 26.44413
  Submission rows : 174722
  Output file     : submission.csv
